# Customer Risk Intelligence
# 04 — Risk, Revenue & Retention Prioritization

This notebook translates calibrated churn probabilities into a business-oriented retention-prioritization framework.

The notebook consumes the **held-out research predictions exported by Notebook 03**. It does not retrain the churn model.

Core flow:

**Calibrated Churn Probability → Revenue at Risk → Revenue Risk Score → Retention Priority → Targeting Strategy**

The analysis includes:
- Revenue at Risk
- Revenue Risk Score
- Retention Priority Score
- Probability-only targeting
- Revenue-aware targeting
- Churn capture
- Financial exposure
- Sensitivity analysis

All targeting results are evaluated on the held-out research test set.


In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("Notebook 04 environment initialized.")


Notebook 04 environment initialized.


In [15]:
# Load the calibrated research predictions produced by Notebook 03

PREDICTIONS_PATH = Path("../data/processed/research_test_predictions.csv")

research_predictions = pd.read_csv(PREDICTIONS_PATH)

print("Research predictions loaded successfully.")
print("Shape:", research_predictions.shape)
print("\nColumns:")
print(research_predictions.columns.tolist())


Research predictions loaded successfully.
Shape: (1409, 28)

Columns:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'AnnualRevenue', 'TenureGroup', 'MonthlyChargeGroup', 'customerID', 'ActualChurn', 'ChurnProbability', 'PredictedChurn', 'CalibratedChurnProbability', 'CalibratedRiskLevel']


In [16]:
# Verify the required research outputs before continuing

required_columns = [
    "customerID",
    "ActualChurn",
    "CalibratedChurnProbability",
    "MonthlyCharges",
    "AnnualRevenue"
]

missing_columns = [
    column for column in required_columns
    if column not in research_predictions.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("Required columns verified.")


Required columns verified.


## 1. Revenue at Risk

For retention prioritization, customer value is incorporated into the churn probability.

We estimate expected **monthly revenue at risk** as:

**Calibrated churn probability × Monthly Charges**

This is an expected exposure measure, not revenue that is guaranteed to be lost.


In [17]:
research_predictions["RevenueAtRisk"] = (
    research_predictions["CalibratedChurnProbability"]
    * research_predictions["MonthlyCharges"]
)

print("=" * 60)
print("REVENUE AT RISK")
print("=" * 60)

print(
    research_predictions[
        [
            "customerID",
            "CalibratedChurnProbability",
            "MonthlyCharges",
            "RevenueAtRisk"
        ]
    ].head(10)
)

print(
    f"\nTotal expected monthly revenue exposure: "
    f"${research_predictions['RevenueAtRisk'].sum():,.2f}"
)


REVENUE AT RISK
   customerID  CalibratedChurnProbability  MonthlyCharges  RevenueAtRisk
0  4376-KFVRS                      0.0547        114.0500         6.2437
1  2754-SDJRD                      0.7041        100.1500        70.5154
2  9917-KWRBE                      0.0859         78.3500         6.7295
3  0365-GXEZS                      0.2726         78.2000        21.3203
4  9385-NXKDA                      0.0586         82.6500         4.8428
5  4686-UXDML                      0.4690         99.8500        46.8252
6  2227-JRSJX                      0.3925         99.1500        38.9192
7  4830-FAXFM                      0.1077         24.1000         2.5966
8  1830-IPXVJ                      0.0570         19.7500         1.1250
9  4690-LLKUA                      0.3924         45.0500        17.6792

Total expected monthly revenue exposure: $27,257.11


In [18]:
# Risk concentration by calibrated risk level

risk_revenue = (
    research_predictions
    .groupby("CalibratedRiskLevel", observed=True)["RevenueAtRisk"]
    .agg(
        TotalRevenueAtRisk="sum",
        AverageRevenueAtRisk="mean",
        Customers="count"
    )
)

risk_revenue["PercentageOfTotalRisk"] = (
    risk_revenue["TotalRevenueAtRisk"]
    / research_predictions["RevenueAtRisk"].sum()
    * 100
)

print(risk_revenue.round(2))


                     TotalRevenueAtRisk  AverageRevenueAtRisk  Customers  \
CalibratedRiskLevel                                                        
High                        10,068.8500               54.4300        185   
Low                          6,963.8500                7.5900        917   
Medium                      10,224.4100               33.3000        307   

                     PercentageOfTotalRisk  
CalibratedRiskLevel                         
High                               36.9400  
Low                                25.5500  
Medium                             37.5100  


## 2. Revenue-Aware Retention Priority

The original project methodology combines:

- **60% churn probability**
- **40% revenue risk**

The revenue-risk component is normalized within the held-out test set. This avoids using information from outside the evaluation population when ranking intervention candidates.


In [19]:
# Revenue Risk Score

max_revenue_risk = research_predictions["RevenueAtRisk"].max()

research_predictions["RevenueRiskScore"] = (
    research_predictions["RevenueAtRisk"]
    / max_revenue_risk
)

# 60% probability + 40% revenue weighting
research_predictions["RetentionPriorityScore"] = (
    0.60 * research_predictions["CalibratedChurnProbability"]
    + 0.40 * research_predictions["RevenueRiskScore"]
)

priority_columns = [
    "customerID",
    "CalibratedChurnProbability",
    "MonthlyCharges",
    "RevenueAtRisk",
    "RevenueRiskScore",
    "RetentionPriorityScore"
]

print(
    research_predictions[
        priority_columns
    ]
    .sort_values("RetentionPriorityScore", ascending=False)
    .head(20)
)


      customerID  CalibratedChurnProbability  MonthlyCharges  RevenueAtRisk  \
1175  6861-XWTWQ                      0.8022         99.2500        79.6152   
647   6023-YEBUP                      0.7891        100.9500        79.6620   
1164  8884-ADFVN                      0.7809        101.9500        79.6145   
634   1751-NCDLI                      0.7918         98.8500        78.2697   
604   4445-ZJNMU                      0.7869         99.3000        78.1355   
947   3006-XIMLN                      0.8016         94.2000        75.5114   
1188  0655-RBDUG                      0.7843         98.0500        76.8997   
1221  0295-PPHDO                      0.7925         95.4500        75.6481   
995   8161-QYMTT                      0.7974         94.1000        75.0362   
627   9057-SIHCH                      0.7880         96.6000        76.1227   
304   6328-ZPBGN                      0.7870         95.1500        74.8863   
788   1143-NMNQJ                      0.8215        

In [20]:
# Assign retention-priority tiers

research_predictions["RetentionPriority"] = pd.cut(
    research_predictions["RetentionPriorityScore"],
    bins=[-0.01, 0.25, 0.50, 0.75, 1.00],
    labels=["Low", "Moderate", "High", "Critical"]
)

print(
    research_predictions["RetentionPriority"]
    .value_counts()
    .sort_index()
)


RetentionPriority
Low         870
Moderate    280
High        187
Critical     72
Name: count, dtype: int64


In [21]:
# Validate retention-priority tiers against observed churn

priority_validation = (
    research_predictions
    .groupby("RetentionPriority", observed=True)["ActualChurn"]
    .agg(
        ChurnRate="mean",
        Churners="sum",
        Customers="count"
    )
)

priority_validation["ChurnRate"] *= 100

print(priority_validation.round(2))


                   ChurnRate  Churners  Customers
RetentionPriority                                
Low                  10.6900        93        870
Moderate             42.5000       119        280
High                 57.2200       107        187
Critical             76.3900        55         72


In [22]:
# Revenue exposure by retention-priority tier

priority_revenue = (
    research_predictions
    .groupby("RetentionPriority", observed=True)["RevenueAtRisk"]
    .agg(
        TotalRevenueAtRisk="sum",
        AverageRevenueAtRisk="mean",
        Customers="count"
    )
)

priority_revenue["PercentageOfTotalRisk"] = (
    priority_revenue["TotalRevenueAtRisk"]
    / research_predictions["RevenueAtRisk"].sum()
    * 100
)

print(priority_revenue.round(2))


                   TotalRevenueAtRisk  AverageRevenueAtRisk  Customers  \
RetentionPriority                                                        
Low                        5,691.2500                6.5400        870   
Moderate                   7,573.3300               27.0500        280   
High                       9,115.7200               48.7500        187   
Critical                   4,876.8100               67.7300         72   

                   PercentageOfTotalRisk  
RetentionPriority                         
Low                              20.8800  
Moderate                         27.7800  
High                             33.4400  
Critical                         17.8900  


In [23]:
# Top critical customers by expected monthly revenue exposure

critical_customers = (
    research_predictions[
        research_predictions["RetentionPriority"] == "Critical"
    ]
    .sort_values("RevenueAtRisk", ascending=False)
)

print(
    critical_customers[
        [
            "customerID",
            "MonthlyCharges",
            "CalibratedChurnProbability",
            "RevenueAtRisk",
            "RetentionPriorityScore",
            "ActualChurn"
        ]
    ]
    .head(20)
    .to_string(index=False)
)


customerID  MonthlyCharges  CalibratedChurnProbability  RevenueAtRisk  RetentionPriorityScore  ActualChurn
6023-YEBUP        100.9500                      0.7891        79.6620                  0.8735            1
6861-XWTWQ         99.2500                      0.8022        79.6152                  0.8811            1
8884-ADFVN        101.9500                      0.7809        79.6145                  0.8683            1
1751-NCDLI         98.8500                      0.7918        78.2697                  0.8681            0
4445-ZJNMU         99.3000                      0.7869        78.1355                  0.8645            0
0655-RBDUG         98.0500                      0.7843        76.8997                  0.8567            1
9057-SIHCH         96.6000                      0.7880        76.1227                  0.8550            1
0295-PPHDO         95.4500                      0.7925        75.6481                  0.8554            1
3006-XIMLN         94.2000           

## 3. Probability-Only vs Revenue-Aware Targeting

A key research question is whether adding customer value changes which customers should receive retention attention.

We compare two strategies at fixed resource levels:

1. **Probability-only:** rank by calibrated churn probability.
2. **Revenue-aware:** rank by the combined Retention Priority Score.

The evaluation measures:
- customers targeted
- observed churners captured
- churner capture percentage
- churners per 100 targeted
- revenue at risk targeted


In [24]:
target_percentages = [0.05, 0.10, 0.15, 0.20]

total_customers = len(research_predictions)
total_churners = research_predictions["ActualChurn"].sum()

probability_ranked = (
    research_predictions
    .sort_values("CalibratedChurnProbability", ascending=False)
    .reset_index(drop=True)
)

priority_ranked = (
    research_predictions
    .sort_values("RetentionPriorityScore", ascending=False)
    .reset_index(drop=True)
)

comparison_results = []

for pct in target_percentages:
    n_target = int(np.ceil(total_customers * pct))

    prob_targeted = probability_ranked.head(n_target)
    priority_targeted = priority_ranked.head(n_target)

    prob_churners = prob_targeted["ActualChurn"].sum()
    priority_churners = priority_targeted["ActualChurn"].sum()

    comparison_results.append({
        "TargetPercentage": pct * 100,
        "CustomersTargeted": n_target,

        "Probability_ChurnersCaptured": int(prob_churners),
        "Probability_CapturePct":
            prob_churners / total_churners * 100,
        "Probability_ChurnersPer100":
            prob_churners / n_target * 100,
        "Probability_RevenueAtRisk":
            prob_targeted["RevenueAtRisk"].sum(),

        "Priority_ChurnersCaptured": int(priority_churners),
        "Priority_CapturePct":
            priority_churners / total_churners * 100,
        "Priority_ChurnersPer100":
            priority_churners / n_target * 100,
        "Priority_RevenueAtRisk":
            priority_targeted["RevenueAtRisk"].sum()
    })

targeting_comparison = pd.DataFrame(comparison_results)

print(
    targeting_comparison.round(2).to_string(index=False)
)


 TargetPercentage  CustomersTargeted  Probability_ChurnersCaptured  Probability_CapturePct  Probability_ChurnersPer100  Probability_RevenueAtRisk  Priority_ChurnersCaptured  Priority_CapturePct  Priority_ChurnersPer100  Priority_RevenueAtRisk
           5.0000                 71                            55                 14.7100                     77.4600                 4,489.0300                         54              14.4400                  76.0600              4,814.4400
          10.0000                141                           106                 28.3400                     75.1800                 8,119.0000                        102              27.2700                  72.3400              8,822.5500
          15.0000                212                           144                 38.5000                     67.9200                11,333.6700                        139              37.1700                  65.5700             12,226.2100
          20.0000           

## 4. Churn Capture vs Financial Exposure Trade-off

In [25]:
tradeoff = targeting_comparison.copy()

tradeoff["ChurnCaptureChangePct"] = (
    (
        tradeoff["Priority_CapturePct"]
        - tradeoff["Probability_CapturePct"]
    )
    / tradeoff["Probability_CapturePct"]
) * 100

tradeoff["RevenueRiskChangePct"] = (
    (
        tradeoff["Priority_RevenueAtRisk"]
        - tradeoff["Probability_RevenueAtRisk"]
    )
    / tradeoff["Probability_RevenueAtRisk"]
) * 100

tradeoff["Probability_RevenueRiskPerCustomer"] = (
    tradeoff["Probability_RevenueAtRisk"]
    / tradeoff["CustomersTargeted"]
)

tradeoff["Priority_RevenueRiskPerCustomer"] = (
    tradeoff["Priority_RevenueAtRisk"]
    / tradeoff["CustomersTargeted"]
)

tradeoff["Probability_RevenueRiskPerChurner"] = (
    tradeoff["Probability_RevenueAtRisk"]
    / tradeoff["Probability_ChurnersCaptured"]
)

tradeoff["Priority_RevenueRiskPerChurner"] = (
    tradeoff["Priority_RevenueAtRisk"]
    / tradeoff["Priority_ChurnersCaptured"]
)

print(
    tradeoff[
        [
            "TargetPercentage",
            "CustomersTargeted",
            "ChurnCaptureChangePct",
            "RevenueRiskChangePct",
            "Probability_RevenueRiskPerCustomer",
            "Priority_RevenueRiskPerCustomer",
            "Probability_RevenueRiskPerChurner",
            "Priority_RevenueRiskPerChurner"
        ]
    ].round(2).to_string(index=False)
)


 TargetPercentage  CustomersTargeted  ChurnCaptureChangePct  RevenueRiskChangePct  Probability_RevenueRiskPerCustomer  Priority_RevenueRiskPerCustomer  Probability_RevenueRiskPerChurner  Priority_RevenueRiskPerChurner
           5.0000                 71                -1.8200                7.2500                             63.2300                          67.8100                            81.6200                         89.1600
          10.0000                141                -3.7700                8.6700                             57.5800                          62.5700                            76.5900                         86.5000
          15.0000                212                -3.4700                7.8800                             53.4600                          57.6700                            78.7100                         87.9600
          20.0000                282                 2.8700                3.7600                             50.5500           

## 5. Sensitivity Analysis

The default retention-priority score uses a 60/40 split between churn probability and revenue risk.

We test alternative weights:

- 50/50
- 60/40
- 70/30
- 80/20

This checks whether the targeting conclusions are highly sensitive to the chosen weighting scheme.


In [26]:
weight_results = []

target_pct = 0.10
n_target = int(np.ceil(len(research_predictions) * target_pct))
total_churners = research_predictions["ActualChurn"].sum()

weight_configs = [
    (0.50, 0.50),
    (0.60, 0.40),
    (0.70, 0.30),
    (0.80, 0.20)
]

for churn_weight, revenue_weight in weight_configs:
    sensitivity_df = research_predictions.copy()

    sensitivity_df["SensitivityScore"] = (
        churn_weight * sensitivity_df["CalibratedChurnProbability"]
        + revenue_weight * sensitivity_df["RevenueRiskScore"]
    )

    sensitivity_df = sensitivity_df.sort_values(
        "SensitivityScore",
        ascending=False
    )

    targeted = sensitivity_df.head(n_target)

    churners_captured = targeted["ActualChurn"].sum()

    weight_results.append({
        "ChurnWeight": churn_weight,
        "RevenueWeight": revenue_weight,
        "CustomersTargeted": n_target,
        "ChurnersCaptured": int(churners_captured),
        "ChurnCapturePct":
            churners_captured / total_churners * 100,
        "ChurnersPer100":
            churners_captured / n_target * 100,
        "RevenueAtRiskTargeted":
            targeted["RevenueAtRisk"].sum(),
        "RevenueRiskPerCustomer":
            targeted["RevenueAtRisk"].sum() / n_target
    })

weight_sensitivity = pd.DataFrame(weight_results)

print(
    weight_sensitivity.round(2).to_string(index=False)
)


 ChurnWeight  RevenueWeight  CustomersTargeted  ChurnersCaptured  ChurnCapturePct  ChurnersPer100  RevenueAtRiskTargeted  RevenueRiskPerCustomer
      0.5000         0.5000                141               103          27.5400         73.0500             8,869.5200                 62.9000
      0.6000         0.4000                141               102          27.2700         72.3400             8,822.5500                 62.5700
      0.7000         0.3000                141               103          27.5400         73.0500             8,766.8300                 62.1800
      0.8000         0.2000                141               103          27.5400         73.0500             8,627.7500                 61.1900


## Conclusion

This notebook extends calibrated churn prediction into a customer-level retention decision framework.

The analysis evaluates customers not only by their probability of churn, but also by the financial exposure associated with that risk. Probability-only and revenue-aware targeting are compared under fixed intervention budgets, followed by sensitivity analysis of the retention-priority weighting.

These results should be interpreted as **decision-support evidence**. Revenue at Risk represents expected exposure, and churn capture on historical held-out data does not prove that an intervention would prevent those customers from churning.

The next stages are dashboarding, application development, and the GenAI/Hugging Face decision-support layer.
